### Step 3 - Novelty Scoring (Unsupervised, Temporal) - Feature Construction Stage

In [1]:
import pandas as pd
import numpy as np
from tqdm import tqdm

- STEP 3B — Citation Features
- Extract:
- cited_by_count
- reference_count

In [2]:
meta = pd.read_csv("../../outputs/intermediate/openalex_metadata_full.csv")

citation_df = meta[[
    "global_paper_id",
    "cited_by_count",
    "referenced_works",
    "year"
]].copy()


def count_refs(x):
    if pd.isna(x):
        return 0
    try:
        return len(eval(x))
    except:
        return 0


citation_df["reference_count"] = citation_df["referenced_works"].apply(count_refs)

citation_df = citation_df.rename(columns={
    "global_paper_id": "paper_id"
})

citation_df = citation_df[[
    "paper_id",
    "year",
    "cited_by_count",
    "reference_count"
]]

citation_df.to_csv("../../outputs/other/citation_features.csv", index=False)

print("Citation features saved.")

Citation features saved.


- STEP 3C — Feature Matrix Construction
- Combine:
- Semantic novelty (kNN)
- Structural novelty
- Citation features

In [3]:
semantic_df = pd.read_csv("../../outputs/final/semantic_novelty_knn_scores.csv")
struct_df = pd.read_csv("../../outputs/final/structural_novelty_scores.csv")
citation_df = pd.read_csv("../../outputs/other/citation_features.csv")

# Merge
features = semantic_df.merge(struct_df,
                             on="paper_id",
                             how="left")

features = features.merge(citation_df,
                          on=["paper_id", "year"],
                          how="left")

features = features.fillna(0)

print("Feature matrix shape:", features.shape)

features.to_csv("../../outputs/other/novelty_feature_matrix.csv", index=False)

Feature matrix shape: (2511, 7)


- STEP 3D — Composite Novelty Score (Continuous)
- Instead of classification, we define:
-   composite_novelty =
-       w1 * structural +
-       w2 * semantic +
-       w3 * citation_signal
- Citation signal normalized.

In [4]:
features = pd.read_csv("../../outputs/other/novelty_feature_matrix.csv")

# Normalize citation signal (log transform for stability)
features["citation_signal"] = np.log1p(features["cited_by_count"])


# Min-max normalize components
def minmax(x):
    return (x - x.min()) / (x.max() - x.min() + 1e-9)


features["struct_norm"] = minmax(features["structural_novelty"])
features["semantic_norm"] = minmax(features["semantic_knn"])
features["citation_norm"] = minmax(features["citation_signal"])

# Weighted combination (can tune later)
features["composite_novelty"] = (
        0.65 * features["struct_norm"] +
        0.25 * features["semantic_norm"] +
        0.15 * features["citation_norm"]
)

features.to_csv("../../outputs/final/novelty_feature_matrix_with_score.csv",
                index=False)

print("Composite novelty score computed.")

Composite novelty score computed.


In [5]:
pd.read_csv('../../outputs/final/novelty_feature_matrix_with_score.csv').columns

Index(['paper_id', 'year', 'semantic_knn', 'structural_novelty',
       'triple_count', 'cited_by_count', 'reference_count', 'citation_signal',
       'struct_norm', 'semantic_norm', 'citation_norm', 'composite_novelty'],
      dtype='object')

In [6]:
cn = pd.read_csv('../../outputs/final/novelty_feature_matrix_with_score.csv')
cn

,paper_id,year,semantic_knn,structural_novelty,triple_count,cited_by_count,reference_count,citation_signal,struct_norm,semantic_norm,citation_norm,composite_novelty
0,SKG_SUM_7,2010,1.000000,1.000000,336,20,26,3.044522,1.000000,1.000000,0.336536,0.950480
1,SKG_MT_1325,2010,1.000000,1.000000,107,34,21,3.555348,1.000000,1.000000,0.393002,0.958950
2,SKG_MT_1231,2010,1.000000,1.000000,59,4,14,1.609438,1.000000,1.000000,0.177904,0.926686
3,SKG_MT_1232,2010,1.000000,1.000000,95,14,13,2.708050,1.000000,1.000000,0.299343,0.944901
4,SKG_MT_388,2010,1.000000,1.000000,61,32,29,3.496508,1.000000,1.000000,0.386498,0.957975
...,...,...,...,...,...,...,...,...,...,...,...,...
2506,NOVEL_MT_244,2022,0.947272,0.692308,117,4,30,1.609438,0.692308,0.877240,0.177904,0.695996
2507,NOVEL_MT_280,2022,0.977790,0.553571,56,10,35,2.397895,0.553571,0.948293,0.265059,0.636653
2508,NOVEL_MT_60,2022,0.941635,0.613333,75,7,0,2.079442,0.613333,0.864117,0.229858,0.649175
2509,NOVEL_MT_306,2024,0.946806,0.674157,89,8,28,2.197225,0.674157,0.876156,0.242877,0.693673


In [7]:
pd.read_csv('../../outputs/final/novelty_feature_matrix_with_score.csv')['composite_novelty'].head()

0    0.950480
1    0.958950
2    0.926686
3    0.944901
4    0.957975
Name: composite_novelty, dtype: float64

In [8]:
features[['semantic_knn', 'structural_novelty']].corr()

,semantic_knn,structural_novelty
semantic_knn,1.000000,-0.094759
structural_novelty,-0.094759,1.000000


In [9]:
print(cn.groupby('year')['triple_count'])

In [10]:
df_clean = cn[cn['year']>2010]
df_clean = df_clean[df_clean['triple_count']>0]

In [11]:
df_clean.groupby('year').size()

year
2011     64
2012     73
2013    122
2014     89
2015    116
2016    129
2017    221
2018    290
2019    436
2020    504
2021    403
2022      3
2024      1
2025      1
dtype: int64

In [13]:
from scipy import stats
r, p = stats.pearsonr(df_clean['semantic_knn'], df_clean['structural_novelty'])

In [14]:
print(f"Clean correlation: r={r:.4f}, p={p:.4f}, n={len(df_clean)}")

Clean correlation: r=-0.2715, p=0.0000, n=2452


In [15]:
cn['composite_novelty']

0       0.950480
1       0.958950
2       0.926686
3       0.944901
4       0.957975
          ...   
2506    0.695996
2507    0.636653
2508    0.649175
2509    0.693673
2510    0.520522
Name: composite_novelty, Length: 2511, dtype: float64